# Analýza nároku na úhradu výdavkov

Tento notebook demonštruje, ako vytvoriť agentov, ktorí používajú pluginy na spracovanie cestovných výdavkov z lokálnych obrázkov blokov, generovanie e-mailu s nárokom na úhradu a vizualizáciu dát o výdavkoch pomocou koláčového grafu. Agenti dynamicky vyberajú funkcie na základe kontextu úlohy.

Kroky:
1. OCR agent spracuje lokálny obrázok bločku a extrahuje údaje o cestovných výdavkoch.
2. Email agent vytvorí e-mail s nárokom na úhradu výdavkov.

### Príklad scenára cestovných výdavkov:
Predstavte si, že ste zamestnanec cestujúci na pracovné stretnutie do iného mesta. Vaša spoločnosť má politiku preplácania všetkých rozumných cestovných výdavkov. Tu je rozpis potenciálnych cestovných výdavkov:
- Doprava:
Letecká doprava na spiatočnú cestu z vášho domovského mesta do cieľového mesta.
Taxi alebo služby zdieľanej jazdy na letisko a späť.
Miestna doprava v cieľovom meste (napríklad verejná doprava, požičovňa áut alebo taxíky).

- Ubytovanie:
Ubytovanie na tri noci v stredne drahom obchodnom hoteli blízko miesta stretnutia.

- Stravovanie:
Denný paušál na raňajky, obed a večeru podľa politiky spoločnosti.

- Rôzne výdavky:
Parkovné na letisku.
Poplatky za prístup na internet v hoteli.
Tipy alebo malé poplatky za služby.

- Dokumentácia:
Predkladáte všetky bločky (letenky, taxi, hotel, strava atď.) a vyplnenú správu o výdavkoch na preplatenie.


## Import potrebných knižníc

Importujte potrebné knižnice a moduly pre notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import Content, Message, tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definujte modely výdavkov

 Vytvorte Pydantic model pre jednotlivé výdavky a triedu ExpenseFormatter na konverziu používateľského dotazu na štruktúrované údaje o výdavkoch.

 Každý výdavok bude reprezentovaný v nasledovnom formáte:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definovanie nástrojov - Generovanie e-mailu

Vytvorte nástrojovú funkciu na generovanie e-mailu na podanie žiadosti o náhradu výdavkov.
- Tento nástroj používa dekorátor `@tool` z Microsoft Agent Framework.
- Vypočíta celkovú sumu výdavkov a naformátuje detaily do tela e-mailu.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Načítavanie obrázkov dokladov o cestovných výdavkoch

Vytvorte pomocnú funkciu na načítanie obrázkov dokladov ako natívne multimodálneho obsahu.
- Načíta bajty obrázkov dokladov a zabalí ich do Agent Framework `Content`, aby ich agent mohol analyzovať.


In [ ]:
def load_receipt_image(image_path: str = "receipt.jpg") -> Content:
    """Load a receipt image as native multimodal content."""
    with open(image_path, "rb") as f:
        image_bytes = f.read()
    return Content.from_data(image_bytes, "image/jpeg")

## Spracovanie výdavkov

Definujte agentov a prepojte ich do sekvenčného pracovného postupu pomocou `WorkflowBuilder`.
- Agent OCR extrahuje štruktúrované údaje o výdavkoch z obrázka pokladničného dokladu, ktorý je priložený v správe používateľa.
- Agent Email vezme extrahované údaje a pomocou nástroja `generate_expense_email` vygeneruje profesionálny e-mail s nárokom na výdavky.
- `WorkflowBuilder` s `add_edge` vytvára sekvenčný pipeline: OCR Agent → Email Agent.


In [ ]:
ocr_agent = client.as_agent(
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Analyze the receipt image supplied in the user message and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hlavná funkcia

Vytvorte sekvenčný pracovný tok a spustite ho na spracovanie obrázka účtenky a vygenerovanie e-mailu s požiadavkou na náhradu výdavkov.


> **Poznámka:** Dáta účtenky sú zabalené v natívnom multimodálnom rámci Agenta `Content`, takže Foundry ich prijíma ako vstupný obrázok, nie ako bežný text.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
receipt_message = Message(
    role="user",
    contents=[prompt, load_receipt_image("receipt.jpg")],
)

events = workflow.run(
    receipt_message,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
